# 11. Optimizer updates — Prodigy, generic Muon, DeepSeek-V4 Muon, K3 Per-Head Muon

Small matrices are used, but the optimizer state and update chains are preserved.


In [ ]:
import math
import torch

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. AdamW reference


In [ ]:
weight = torch.tensor([[1.0, -1.0], [0.5, 2.0]], device=device)
gradient = torch.tensor([[0.2, -0.4], [1.0, 0.5]], device=device)
m = torch.zeros_like(weight)
v = torch.zeros_like(weight)

for step in range(1, 4):
    m = 0.9 * m + 0.1 * gradient
    v = 0.999 * v + 0.001 * gradient.square()
    m_hat = m / (1 - 0.9 ** step)
    v_hat = v / (1 - 0.999 ** step)
    weight = weight * (1 - 0.1 * 0.01) - 0.1 * m_hat / (v_hat.sqrt() + 1e-8)
print("AdamW:", weight)


## 2. Prodigy D-adaptation


In [ ]:
class TinyProdigyState:
    def __init__(self, parameter):
        self.p0 = parameter.clone()
        self.s = torch.zeros_like(parameter)
        self.exp_avg = torch.zeros_like(parameter)
        self.exp_avg_sq = torch.zeros_like(parameter)
        self.d = 1e-3
        self.d_max = self.d
        self.d_numerator = 0.0


def prodigy_step(parameter, gradient, state, beta1=0.9, beta2=0.999):
    beta3 = math.sqrt(beta2)
    d0 = 1e-3
    adapted_lr = state.d
    displacement = state.p0 - parameter

    delta_numerator = (state.d / d0) * adapted_lr * torch.sum(gradient * displacement).item()
    state.d_numerator = beta3 * state.d_numerator + delta_numerator
    state.s = beta3 * state.s + ((state.d / d0) * adapted_lr) * gradient

    denominator = state.s.abs().sum().item()
    if denominator > 0:
        d_hat = state.d_numerator / denominator
        state.d_max = max(state.d_max, d_hat)
        state.d = max(state.d, state.d_max)

    state.exp_avg = beta1 * state.exp_avg + state.d * (1 - beta1) * gradient
    state.exp_avg_sq = beta2 * state.exp_avg_sq + state.d**2 * (1 - beta2) * gradient.square()
    update = state.exp_avg / (state.exp_avg_sq.sqrt() + state.d * 1e-8)
    return parameter - update


parameter = torch.tensor([[1.0, -1.0], [0.5, 2.0]], device=device)
state = TinyProdigyState(parameter)
for scale in [1.0, 0.7, 0.4]:
    parameter = prodigy_step(parameter, scale * gradient, state)
    print("Prodigy d:", state.d)


## 3. Newton-Schulz polynomial primitive


In [ ]:
def ns_polynomial(x, coefficients):
    a, b, c = coefficients
    gram = x @ x.mT
    return a * x + (b * gram + c * (gram @ gram)) @ x


def normalize_matrix(matrix):
    x = matrix.float()
    transposed = x.size(-2) > x.size(-1)
    if transposed:
        x = x.mT
    x = x / (x.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    return x, transposed


## 4. Generic Muon quintic direction


In [ ]:
def generic_muon_orthogonalize(matrix, steps=5):
    x, transposed = normalize_matrix(matrix)
    coefficients = (3.4445, -4.7750, 2.0315)
    for _ in range(steps):
        x = ns_polynomial(x, coefficients)
    if transposed:
        x = x.mT
    return x.to(matrix.dtype)


def nesterov_momentum(gradient, momentum_buffer, beta=0.95):
    momentum_buffer.mul_(beta).add_(gradient, alpha=1 - beta)
    return beta * momentum_buffer + (1 - beta) * gradient


def generic_muon_direction(gradient, momentum_buffer, beta=0.95):
    raw_update = nesterov_momentum(gradient, momentum_buffer, beta)
    orthogonal = generic_muon_orthogonalize(raw_update)
    scale = math.sqrt(max(gradient.size(-2), gradient.size(-1)))
    return scale * orthogonal


## 5. DeepSeek-V4 hybrid Muon

V4 uses ten Newton-Schulz iterations: eight aggressive iterations with `(3.4445, -4.7750, 2.0315)`, followed by two stabilizing iterations with `(2, -1.5, 0.5)`. The orthogonalized matrix is RMS-rescaled by `gamma=0.18`, then decoupled weight decay and the parameter update are applied. Embeddings, prediction heads, RMSNorm and mHC gating/static-bias parameters remain on AdamW instead of this matrix path.


In [ ]:
def deepseek_v4_hybrid_ns(matrix):
    x, transposed = normalize_matrix(matrix)

    fast = (3.4445, -4.7750, 2.0315)
    stable = (2.0, -1.5, 0.5)

    for _ in range(8):
        x = ns_polynomial(x, fast)
    for _ in range(2):
        x = ns_polynomial(x, stable)

    if transposed:
        x = x.mT
    return x.to(matrix.dtype)


def deepseek_v4_muon_step(
    weight,
    gradient,
    momentum_buffer,
    learning_rate=0.02,
    momentum=0.95,
    weight_decay=0.1,
    gamma=0.18,
):
    nesterov_update = nesterov_momentum(
        gradient,
        momentum_buffer,
        beta=momentum,
    )
    orthogonal = deepseek_v4_hybrid_ns(nesterov_update)

    rows, columns = gradient.shape
    rms_rescale = gamma * math.sqrt(max(rows, columns))
    direction = rms_rescale * orthogonal

    updated = (
        weight * (1 - learning_rate * weight_decay)
        - learning_rate * direction
    )
    return updated, direction


v4_weight = torch.randn(16, 12, device=device)
v4_gradient = torch.randn_like(v4_weight)
v4_momentum = torch.zeros_like(v4_weight)
v4_updated, v4_direction = deepseek_v4_muon_step(
    v4_weight,
    v4_gradient,
    v4_momentum,
)
print("V4 Muon delta:", (v4_updated - v4_weight).norm().item())
print("V4 direction RMS:", v4_direction.square().mean().sqrt().item())


## 6. Kimi-K3 Per-Head Muon

K3 partitions attention projection matrices by head and applies the matrix orthogonalization independently to each head instead of flattening all heads into one matrix.


In [ ]:
def per_head_muon_direction(gradient, momentum_buffer, num_heads, beta=0.95):
    out_features, _ = gradient.shape
    assert out_features % num_heads == 0
    rows_per_head = out_features // num_heads
    updates = []

    for head_index in range(num_heads):
        start = head_index * rows_per_head
        stop = start + rows_per_head
        head_gradient = gradient[start:stop]
        head_momentum = momentum_buffer[start:stop]
        raw = nesterov_momentum(head_gradient, head_momentum, beta)
        orthogonal = generic_muon_orthogonalize(raw)
        scale = math.sqrt(max(head_gradient.shape))
        updates.append(scale * orthogonal)

    return torch.cat(updates, dim=0)


num_heads = 4
qkv_gradient = torch.randn(num_heads * 8, 24, device=device)
qkv_momentum = torch.zeros_like(qkv_gradient)

global_update = generic_muon_direction(qkv_gradient, torch.zeros_like(qkv_gradient))
per_head_update = per_head_muon_direction(qkv_gradient, qkv_momentum, num_heads)
print("global/per-head difference:", (global_update - per_head_update).norm().item())
for head_index in range(num_heads):
    rows = slice(head_index * 8, (head_index + 1) * 8)
    print(f"head {head_index} norm:", per_head_update[rows].norm().item())


## References and provenance

- Prodigy: D-adaptation state and parameter-displacement statistics.
- Muon: Nesterov momentum followed by Newton-Schulz orthogonalization.
- DeepSeek-V4: 8 fast + 2 stable hybrid Newton-Schulz iterations, RMS rescaling, decoupled weight decay; non-matrix/special parameter groups stay on AdamW.
- Kimi-K3: Per-Head Muon for attention projections.
